In [1]:
# Cell 1. Stage 1 hyperparameter stability setup
# 원하는 결과:
# - Stage 1 final tensor를 로드한다.
# - 기존 Stage 1 single-seed / seed robustness / Stage 2 / Stage 2B 결과를 reference로 로드한다.
# - 안정화용 MLP hyperparameter grid를 정의한다.
# - 아직 학습은 하지 않는다.

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

STAGE1_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage1"
STAGE1_TENSOR_DIR = STAGE1_DIR / "mlp_current_day_final"
STAGE1_METADATA_PATH = STAGE1_DIR / "metadata.json"
STAGE1_FEATURE_COLUMNS_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"

STAGE1_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_mlp_outputs"
STAGE1_METRICS_PATH = STAGE1_OUTPUT_DIR / "pre_sleep_stage1_mlp_metrics.csv"

STAGE1_SEED_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_mlp_seed_robustness_outputs"
STAGE1_SEED_SUMMARY_PATH = STAGE1_SEED_OUTPUT_DIR / "pre_sleep_stage1_seed_robustness_summary.csv"

STAGE2_METRICS_PATH = (
    PROCESSED_DIR
    / "pre_sleep_forecasting"
    / "design_c_stage2_rolling_history"
    / "experiments"
    / "stage2_mlp_outputs"
    / "pre_sleep_stage2_mlp_metrics.csv"
)

STAGE2B_METRICS_PATH = (
    PROCESSED_DIR
    / "pre_sleep_forecasting"
    / "design_c_stage2b_compact_rolling"
    / "experiments"
    / "stage2b_mlp_outputs"
    / "pre_sleep_stage2b_mlp_metrics.csv"
)

OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_hyperparameter_stability_outputs"
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAGE1_HYPERPARAMETER_STABILITY = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FEATURE_TIMING = "pre_sleep"
SUBSET_NAME = "design_c_stage1_intraday_previous_day"
MODEL_FAMILY = "mlp_current_day"
WINDOW = 1

STABILITY_SEEDS = [7, 123, 2026, 2027]

HYPERPARAMETER_GRID = [
    {
        "config_id": "small_dropout40_wd1e3",
        "hidden_dims": (32, 16),
        "dropout": 0.40,
        "weight_decay": 1e-3,
        "learning_rate": 8e-4,
        "batch_size": 64,
        "max_epochs": 120,
        "patience": 20,
    },
    {
        "config_id": "small_dropout50_wd1e3",
        "hidden_dims": (32, 16),
        "dropout": 0.50,
        "weight_decay": 1e-3,
        "learning_rate": 8e-4,
        "batch_size": 64,
        "max_epochs": 120,
        "patience": 20,
    },
    {
        "config_id": "tiny_dropout40_wd1e3",
        "hidden_dims": (24, 12),
        "dropout": 0.40,
        "weight_decay": 1e-3,
        "learning_rate": 8e-4,
        "batch_size": 64,
        "max_epochs": 120,
        "patience": 20,
    },
    {
        "config_id": "stage1_like_stronger_wd",
        "hidden_dims": (64, 32),
        "dropout": 0.35,
        "weight_decay": 1e-3,
        "learning_rate": 8e-4,
        "batch_size": 64,
        "max_epochs": 120,
        "patience": 20,
    },
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_npz_split(split_name):
    path = STAGE1_TENSOR_DIR / f"{split_name}.npz"
    data = np.load(path, allow_pickle=True)
    return {
        "path": path,
        "X": data["X"].astype(np.float32),
        "y": data["y"].astype(np.int64),
        "sleep_episode_id": data["sleep_episode_id"].astype(str),
        "participant_object_id": data["participant_object_id"].astype(str),
        "feature_columns": data["feature_columns"].astype(str),
    }

split_data = {
    split_name: load_npz_split(split_name)
    for split_name in ["train", "validation", "test"]
}

metadata = json.loads(STAGE1_METADATA_PATH.read_text(encoding="utf-8"))
feature_columns_df = pd.read_csv(STAGE1_FEATURE_COLUMNS_PATH, encoding="utf-8-sig")
stage1_metrics_df = pd.read_csv(STAGE1_METRICS_PATH, encoding="utf-8-sig")
stage1_seed_summary_df = pd.read_csv(STAGE1_SEED_SUMMARY_PATH, encoding="utf-8-sig")
stage2_metrics_df = pd.read_csv(STAGE2_METRICS_PATH, encoding="utf-8-sig")
stage2b_metrics_df = pd.read_csv(STAGE2B_METRICS_PATH, encoding="utf-8-sig")

tensor_summary_rows = []

for split_name, data in split_data.items():
    X = data["X"]
    y = data["y"]

    tensor_summary_rows.append(
        {
            "split": split_name,
            "X_shape": X.shape,
            "samples": int(X.shape[0]),
            "features": int(X.shape[1]),
            "participants": int(pd.Series(data["participant_object_id"]).nunique()),
            "target_0": int((y == 0).sum()),
            "target_1": int((y == 1).sum()),
            "target_mean": float(y.mean()),
            "nan_count": int(np.isnan(X).sum()),
            "inf_count": int(np.isinf(X).sum()),
        }
    )

tensor_summary_df = pd.DataFrame(tensor_summary_rows)

reference_rows = []

stage1_test = stage1_metrics_df[stage1_metrics_df["split"] == "test"].iloc[0]
stage2_test = stage2_metrics_df[stage2_metrics_df["split"] == "test"].iloc[0]
stage2b_test = stage2b_metrics_df[stage2b_metrics_df["split"] == "test"].iloc[0]
stage1_seed_summary = stage1_seed_summary_df.iloc[0]

reference_rows.append(
    {
        "reference": "stage1_single_seed",
        "balanced_accuracy": stage1_test["balanced_accuracy"],
        "roc_auc": stage1_test["roc_auc"],
        "average_precision": stage1_test["average_precision"],
        "f1": stage1_test["f1"],
        "precision": stage1_test["precision"],
        "recall": stage1_test["recall"],
        "note": "best single Stage 1 run so far",
    }
)

reference_rows.append(
    {
        "reference": "stage1_seed_mean",
        "balanced_accuracy": stage1_seed_summary["mean_test_balanced_accuracy"],
        "roc_auc": stage1_seed_summary["mean_test_roc_auc"],
        "average_precision": stage1_seed_summary["mean_test_average_precision"],
        "f1": stage1_seed_summary["mean_test_f1"],
        "precision": stage1_seed_summary["mean_test_precision"],
        "recall": stage1_seed_summary["mean_test_recall"],
        "note": "more reliable Stage 1 robustness reference",
    }
)

reference_rows.append(
    {
        "reference": "stage2_full_rolling",
        "balanced_accuracy": stage2_test["balanced_accuracy"],
        "roc_auc": stage2_test["roc_auc"],
        "average_precision": stage2_test["average_precision"],
        "f1": stage2_test["f1"],
        "precision": stage2_test["precision"],
        "recall": stage2_test["recall"],
        "note": "rolling did not improve generalization",
    }
)

reference_rows.append(
    {
        "reference": "stage2b_compact_rolling",
        "balanced_accuracy": stage2b_test["balanced_accuracy"],
        "roc_auc": stage2b_test["roc_auc"],
        "average_precision": stage2b_test["average_precision"],
        "f1": stage2b_test["f1"],
        "precision": stage2b_test["precision"],
        "recall": stage2b_test["recall"],
        "note": "compact rolling also did not improve BA",
    }
)

reference_df = pd.DataFrame(reference_rows)
hyperparameter_grid_df = pd.DataFrame(HYPERPARAMETER_GRID)

print("=== Stage 1 Hyperparameter Stability Setup ===")
print("DEVICE:", DEVICE)
print("RUN_STAGE1_HYPERPARAMETER_STABILITY:", RUN_STAGE1_HYPERPARAMETER_STABILITY)
print("STAGE1_TENSOR_DIR:", STAGE1_TENSOR_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("seeds:", STABILITY_SEEDS)

print("\n=== Tensor Summary ===")
display(tensor_summary_df)

print("\n=== Feature Group Counts ===")
display(feature_columns_df["feature_group"].value_counts().reset_index(name="features"))

print("\n=== Reference Results ===")
display(reference_df)

print("\n=== Hyperparameter Grid ===")
display(hyperparameter_grid_df)

print("\n=== Validation Checks ===")
print("grid runs:", len(HYPERPARAMETER_GRID) * len(STABILITY_SEEDS))
print("feature count:", split_data["train"]["X"].shape[1])
print("problem rows:", len(tensor_summary_df[(tensor_summary_df["nan_count"] > 0) | (tensor_summary_df["inf_count"] > 0)]))

=== Stage 1 Hyperparameter Stability Setup ===
DEVICE: cpu
RUN_STAGE1_HYPERPARAMETER_STABILITY: False
STAGE1_TENSOR_DIR: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day_final
OUTPUT_DIR: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_hyperparameter_stability_outputs
seeds: [7, 123, 2026, 2027]

=== Tensor Summary ===


,split,X_shape,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count
0,train,"(2323, 58)",2323,58,46,1365,958,0.412398,0,0
1,validation,"(347, 58)",347,58,9,225,122,0.351585,0,0
2,test,"(881, 58)",881,58,14,563,318,0.360953,0,0



=== Feature Group Counts ===


,feature_group,features
0,missing_indicator,25
1,pre_sleep_intraday,18
2,previous_day_daily,9
3,timing,6



=== Reference Results ===


,reference,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,note
0,stage1_single_seed,0.633762,0.687501,0.600856,0.490421,0.627451,0.402516,best single Stage 1 run so far
1,stage1_seed_mean,0.610746,0.668134,0.601623,0.530948,0.494221,0.620545,more reliable Stage 1 robustness reference
2,stage2_full_rolling,0.602486,0.662779,0.585455,0.530719,0.454139,0.638365,rolling did not improve generalization
3,stage2b_compact_rolling,0.592293,0.685227,0.578818,0.553005,0.423786,0.795597,compact rolling also did not improve BA



=== Hyperparameter Grid ===


,config_id,hidden_dims,dropout,weight_decay,learning_rate,batch_size,max_epochs,patience
0,small_dropout40_wd1e3,"(32, 16)",0.40,0.001,0.0008,64,120,20
1,small_dropout50_wd1e3,"(32, 16)",0.50,0.001,0.0008,64,120,20
2,tiny_dropout40_wd1e3,"(24, 12)",0.40,0.001,0.0008,64,120,20
3,stage1_like_stronger_wd,"(64, 32)",0.35,0.001,0.0008,64,120,20



=== Validation Checks ===
grid runs: 16
feature count: 58
problem rows: 0


In [3]:
# Cell 2. Run Stage 1 hyperparameter stability grid
# 원하는 결과:
# - Stage 1 feature set에서 MLP hyperparameter grid x seed 반복 학습을 수행한다.
# - validation-selected threshold 기준 train/validation/test metrics를 저장한다.
# - config별 test 성능 평균/표준편차를 요약한다.
# - 기존 Stage 1 seed mean/reference와 비교한다.
# - log/YYYY-MM-DD.md를 업데이트한다.

RUN_STAGE1_HYPERPARAMETER_STABILITY = True

METRICS_PATH = OUTPUT_DIR / "stage1_hyperparameter_stability_metrics.csv"
HISTORY_PATH = OUTPUT_DIR / "stage1_hyperparameter_stability_history.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "stage1_hyperparameter_stability_predictions.csv"
SUMMARY_PATH = OUTPUT_DIR / "stage1_hyperparameter_stability_summary.csv"
COMPARISON_PATH = OUTPUT_DIR / "stage1_hyperparameter_stability_reference_comparison.csv"

class StabilityMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(32, 16), dropout=0.4):
        super().__init__()

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(current_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

def make_loader(X, y, batch_size=64, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def predict_proba(model, X, batch_size=256):
    model.eval()
    probabilities = []

    loader = make_loader(X, np.zeros(X.shape[0]), batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch_X, _ in loader:
            batch_X = batch_X.to(DEVICE)
            logits = model(batch_X)
            batch_probabilities = torch.sigmoid(logits).detach().cpu().numpy()
            probabilities.append(batch_probabilities)

    return np.concatenate(probabilities)

def evaluate_binary_predictions(y_true, y_probability, threshold):
    y_pred = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_probability)
        metrics["average_precision"] = average_precision_score(y_true, y_probability)
    else:
        metrics["roc_auc"] = np.nan
        metrics["average_precision"] = np.nan

    return metrics

def find_best_threshold(y_true, y_probability, metric_name="balanced_accuracy"):
    threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []
    for threshold in threshold_grid:
        row = evaluate_binary_predictions(y_true, y_probability, threshold)
        rows.append(row)

    threshold_df = pd.DataFrame(rows)

    best_row = (
        threshold_df
        .sort_values(
            [metric_name, "f1", "balanced_accuracy"],
            ascending=[False, False, False],
        )
        .iloc[0]
        .to_dict()
    )

    return float(best_row["threshold"]), threshold_df, best_row

def train_one_stability_model(
    experiment_id,
    config,
    seed,
    X_train,
    y_train,
    X_validation,
    y_validation,
    input_dim,
):
    set_seed(seed)

    model = StabilityMLP(
        input_dim=input_dim,
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
    ).to(DEVICE)

    positive_count = float((y_train == 1).sum())
    negative_count = float((y_train == 0).sum())
    pos_weight = torch.tensor(
        [negative_count / max(positive_count, 1.0)],
        dtype=torch.float32,
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    train_loader = make_loader(
        X_train,
        y_train,
        batch_size=config["batch_size"],
        shuffle=True,
    )

    best_state = None
    best_validation_balanced_accuracy = -np.inf
    best_epoch = None
    best_threshold = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        validation_probability = predict_proba(model, X_validation)
        threshold, threshold_df, best_threshold_row = find_best_threshold(
            y_validation,
            validation_probability,
            metric_name="balanced_accuracy",
        )

        validation_metrics = evaluate_binary_predictions(
            y_validation,
            validation_probability,
            threshold,
        )

        mean_train_loss = float(np.mean(train_losses))

        history_rows.append(
            {
                "experiment_id": experiment_id,
                "config_id": config["config_id"],
                "seed": seed,
                "epoch": epoch,
                "train_loss": mean_train_loss,
                "validation_selected_threshold": threshold,
                **{
                    f"validation_{key}": value
                    for key, value in validation_metrics.items()
                },
            }
        )

        current_validation_balanced_accuracy = validation_metrics["balanced_accuracy"]

        if current_validation_balanced_accuracy > best_validation_balanced_accuracy:
            best_validation_balanced_accuracy = current_validation_balanced_accuracy
            best_epoch = epoch
            best_threshold = threshold
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"{experiment_id} epoch {epoch:03d} | "
                f"loss={mean_train_loss:.4f} | "
                f"val_bal_acc={validation_metrics['balanced_accuracy']:.4f} | "
                f"val_auc={validation_metrics['roc_auc']:.4f} | "
                f"thr={threshold:.2f}"
            )

        if epochs_without_improvement >= config["patience"]:
            print(
                f"{experiment_id} early stopped at epoch {epoch}; "
                f"best_epoch={best_epoch}, "
                f"best_val_bal_acc={best_validation_balanced_accuracy:.4f}"
            )
            break

    model.load_state_dict(best_state)

    return {
        "model": model,
        "history_df": pd.DataFrame(history_rows),
        "best_epoch": best_epoch,
        "best_threshold": best_threshold,
        "best_validation_balanced_accuracy": best_validation_balanced_accuracy,
    }

if not RUN_STAGE1_HYPERPARAMETER_STABILITY:
    print("RUN_STAGE1_HYPERPARAMETER_STABILITY is False.")
    print("Set RUN_STAGE1_HYPERPARAMETER_STABILITY = True to run 16 training jobs manually.")
else:
    all_metrics = []
    all_history = []
    all_predictions = []

    total_runs = len(HYPERPARAMETER_GRID) * len(STABILITY_SEEDS)
    run_index = 0

    print("=== Stage 1 Hyperparameter Stability Grid Started ===")
    print("total runs:", total_runs)
    print("device:", DEVICE)

    for config in HYPERPARAMETER_GRID:
        for seed in STABILITY_SEEDS:
            run_index += 1
            experiment_id = f"presleep_stage1_hp_{config['config_id']}_seed{seed}"
            model_path = MODEL_DIR / f"{experiment_id}_best.pt"

            print("\n" + "=" * 80)
            print(f"Run {run_index}/{total_runs}: {experiment_id}")
            print("=" * 80)

            train_result = train_one_stability_model(
                experiment_id=experiment_id,
                config=config,
                seed=seed,
                X_train=split_data["train"]["X"],
                y_train=split_data["train"]["y"],
                X_validation=split_data["validation"]["X"],
                y_validation=split_data["validation"]["y"],
                input_dim=split_data["train"]["X"].shape[1],
            )

            model = train_result["model"]
            best_epoch = train_result["best_epoch"]
            selected_threshold = train_result["best_threshold"]
            history_df = train_result["history_df"]

            torch.save(
                {
                    "experiment_id": experiment_id,
                    "config_id": config["config_id"],
                    "model_family": MODEL_FAMILY,
                    "feature_timing": FEATURE_TIMING,
                    "subset_name": SUBSET_NAME,
                    "window": WINDOW,
                    "seed": seed,
                    "input_dim": split_data["train"]["X"].shape[1],
                    "hidden_dims": config["hidden_dims"],
                    "dropout": config["dropout"],
                    "weight_decay": config["weight_decay"],
                    "learning_rate": config["learning_rate"],
                    "best_epoch": best_epoch,
                    "selected_threshold_from_validation": selected_threshold,
                    "state_dict": model.state_dict(),
                    "feature_columns": split_data["train"]["feature_columns"].tolist(),
                },
                model_path,
            )

            metric_rows = []
            prediction_rows = []

            for split_name in ["train", "validation", "test"]:
                X_split = split_data[split_name]["X"]
                y_split = split_data[split_name]["y"]
                probability = predict_proba(model, X_split)

                metrics = evaluate_binary_predictions(
                    y_split,
                    probability,
                    selected_threshold,
                )

                metric_rows.append(
                    {
                        "experiment_id": experiment_id,
                        "config_id": config["config_id"],
                        "feature_timing": FEATURE_TIMING,
                        "subset_name": SUBSET_NAME,
                        "model_family": MODEL_FAMILY,
                        "window": WINDOW,
                        "split": split_name,
                        "seed": seed,
                        "hidden_dims": str(config["hidden_dims"]),
                        "dropout": config["dropout"],
                        "weight_decay": config["weight_decay"],
                        "learning_rate": config["learning_rate"],
                        "batch_size": config["batch_size"],
                        "best_epoch": best_epoch,
                        "selected_threshold_from_validation": selected_threshold,
                        **metrics,
                        "model_path": str(model_path.relative_to(PROJECT_ROOT)),
                    }
                )

                prediction_rows.append(
                    pd.DataFrame(
                        {
                            "experiment_id": experiment_id,
                            "config_id": config["config_id"],
                            "split": split_name,
                            "seed": seed,
                            "sleep_episode_id": split_data[split_name]["sleep_episode_id"],
                            "participant_object_id": split_data[split_name]["participant_object_id"],
                            "y_true": y_split,
                            "y_probability": probability,
                            "selected_threshold_from_validation": selected_threshold,
                            "y_pred": (probability >= selected_threshold).astype(int),
                        }
                    )
                )

            all_metrics.append(pd.DataFrame(metric_rows))
            all_history.append(history_df)
            all_predictions.append(pd.concat(prediction_rows, ignore_index=True))

    stability_metrics_df = pd.concat(all_metrics, ignore_index=True)
    stability_history_df = pd.concat(all_history, ignore_index=True)
    stability_predictions_df = pd.concat(all_predictions, ignore_index=True)

    test_metrics_df = stability_metrics_df[
        stability_metrics_df["split"] == "test"
    ].copy()

    stability_summary_df = (
        test_metrics_df
        .groupby(["config_id", "hidden_dims", "dropout", "weight_decay", "learning_rate", "batch_size"])
        .agg(
            runs=("experiment_id", "nunique"),
            mean_test_balanced_accuracy=("balanced_accuracy", "mean"),
            std_test_balanced_accuracy=("balanced_accuracy", "std"),
            min_test_balanced_accuracy=("balanced_accuracy", "min"),
            max_test_balanced_accuracy=("balanced_accuracy", "max"),
            mean_test_roc_auc=("roc_auc", "mean"),
            std_test_roc_auc=("roc_auc", "std"),
            mean_test_average_precision=("average_precision", "mean"),
            mean_test_f1=("f1", "mean"),
            mean_test_precision=("precision", "mean"),
            mean_test_recall=("recall", "mean"),
            mean_selected_threshold=("selected_threshold_from_validation", "mean"),
            mean_best_epoch=("best_epoch", "mean"),
        )
        .reset_index()
        .sort_values(
            ["mean_test_balanced_accuracy", "mean_test_roc_auc"],
            ascending=[False, False],
        )
    )

    stage1_seed_mean_ba = float(
        reference_df.loc[
            reference_df["reference"] == "stage1_seed_mean",
            "balanced_accuracy",
        ].iloc[0]
    )
    stage1_single_seed_ba = float(
        reference_df.loc[
            reference_df["reference"] == "stage1_single_seed",
            "balanced_accuracy",
        ].iloc[0]
    )

    stability_summary_df["delta_mean_ba_vs_stage1_seed_mean"] = (
        stability_summary_df["mean_test_balanced_accuracy"] - stage1_seed_mean_ba
    )
    stability_summary_df["delta_mean_ba_vs_stage1_single_seed"] = (
        stability_summary_df["mean_test_balanced_accuracy"] - stage1_single_seed_ba
    )

    best_config_row = stability_summary_df.iloc[0].copy()

    reference_comparison_df = pd.concat(
        [
            reference_df.assign(row_type="existing_reference"),
            pd.DataFrame(
                [
                    {
                        "reference": f"best_hp_config__{best_config_row['config_id']}",
                        "balanced_accuracy": best_config_row["mean_test_balanced_accuracy"],
                        "roc_auc": best_config_row["mean_test_roc_auc"],
                        "average_precision": best_config_row["mean_test_average_precision"],
                        "f1": best_config_row["mean_test_f1"],
                        "precision": best_config_row["mean_test_precision"],
                        "recall": best_config_row["mean_test_recall"],
                        "note": "mean across stability seeds",
                        "row_type": "new_best_hyperparameter_mean",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    stability_metrics_df.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")
    stability_history_df.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
    stability_predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
    stability_summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
    reference_comparison_df.to_csv(COMPARISON_PATH, index=False, encoding="utf-8-sig")

    metadata["stage1_hyperparameter_stability"] = {
        "seeds": STABILITY_SEEDS,
        "grid_configs": HYPERPARAMETER_GRID,
        "metrics_path": str(METRICS_PATH.relative_to(PROJECT_ROOT)),
        "history_path": str(HISTORY_PATH.relative_to(PROJECT_ROOT)),
        "predictions_path": str(PREDICTIONS_PATH.relative_to(PROJECT_ROOT)),
        "summary_path": str(SUMMARY_PATH.relative_to(PROJECT_ROOT)),
        "comparison_path": str(COMPARISON_PATH.relative_to(PROJECT_ROOT)),
        "best_config_id": best_config_row["config_id"],
        "best_mean_test_balanced_accuracy": float(best_config_row["mean_test_balanced_accuracy"]),
    }

    STAGE1_METADATA_PATH.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    log_entry = f"""

## 2026-06-29 - Pre-sleep Stage 1 hyperparameter stability grid

Ran Stage 1 MLP hyperparameter stability grid.

- Notebook: `notebooks/16_pre_sleep_stage1_hyperparameter_stability.ipynb`
- Feature set: strict pre-sleep Design C Stage 1
- Runs: {total_runs}
- Seeds: {STABILITY_SEEDS}
- Metrics: `{METRICS_PATH.relative_to(PROJECT_ROOT)}`
- History: `{HISTORY_PATH.relative_to(PROJECT_ROOT)}`
- Predictions: `{PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}`
- Summary: `{SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- Reference comparison: `{COMPARISON_PATH.relative_to(PROJECT_ROOT)}`
- Best config by mean test balanced accuracy: `{best_config_row["config_id"]}`
- Best mean test balanced accuracy: {best_config_row["mean_test_balanced_accuracy"]:.4f}
- Delta vs Stage 1 seed mean: {best_config_row["delta_mean_ba_vs_stage1_seed_mean"]:.4f}
- Delta vs Stage 1 single seed: {best_config_row["delta_mean_ba_vs_stage1_single_seed"]:.4f}
"""

    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(log_entry)

    print("\n=== Stage 1 Hyperparameter Stability Grid Saved ===")
    print("metrics:", METRICS_PATH.relative_to(PROJECT_ROOT), METRICS_PATH.exists())
    print("history:", HISTORY_PATH.relative_to(PROJECT_ROOT), HISTORY_PATH.exists())
    print("predictions:", PREDICTIONS_PATH.relative_to(PROJECT_ROOT), PREDICTIONS_PATH.exists())
    print("summary:", SUMMARY_PATH.relative_to(PROJECT_ROOT), SUMMARY_PATH.exists())
    print("comparison:", COMPARISON_PATH.relative_to(PROJECT_ROOT), COMPARISON_PATH.exists())

    print("\n=== Stability Summary Ranked ===")
    display(stability_summary_df)

    print("\n=== Test Metrics By Run ===")
    display(test_metrics_df.sort_values(["config_id", "seed"]))

    print("\n=== Reference Comparison ===")
    display(reference_comparison_df)

    print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Stage 1 Hyperparameter Stability Grid Started ===
total runs: 16
device: cpu

Run 1/16: presleep_stage1_hp_small_dropout40_wd1e3_seed7
presleep_stage1_hp_small_dropout40_wd1e3_seed7 epoch 001 | loss=0.8352 | val_bal_acc=0.6359 | val_auc=0.6466 | thr=0.52
presleep_stage1_hp_small_dropout40_wd1e3_seed7 epoch 010 | loss=0.7428 | val_bal_acc=0.6648 | val_auc=0.6945 | thr=0.52
presleep_stage1_hp_small_dropout40_wd1e3_seed7 epoch 020 | loss=0.7316 | val_bal_acc=0.6662 | val_auc=0.6891 | thr=0.54
presleep_stage1_hp_small_dropout40_wd1e3_seed7 epoch 030 | loss=0.7062 | val_bal_acc=0.6536 | val_auc=0.6805 | thr=0.55
presleep_stage1_hp_small_dropout40_wd1e3_seed7 early stopped at epoch 34; best_epoch=14, best_val_bal_acc=0.6722

Run 2/16: presleep_stage1_hp_small_dropout40_wd1e3_seed123
presleep_stage1_hp_small_dropout40_wd1e3_seed123 epoch 001 | loss=0.8282 | val_bal_acc=0.5977 | val_auc=0.6224 | thr=0.49
presleep_stage1_hp_small_dropout40_wd1e3_seed123 epoch 010 | loss=0.7621 | val_bal_acc

,config_id,hidden_dims,dropout,weight_decay,learning_rate,batch_size,runs,mean_test_balanced_accuracy,std_test_balanced_accuracy,min_test_balanced_accuracy,...,mean_test_roc_auc,std_test_roc_auc,mean_test_average_precision,mean_test_f1,mean_test_precision,mean_test_recall,mean_selected_threshold,mean_best_epoch,delta_mean_ba_vs_stage1_seed_mean,delta_mean_ba_vs_stage1_single_seed
3,tiny_dropout40_wd1e3,"(24, 12)",0.40,0.001,0.0008,64,4,0.658607,0.007761,0.649209,...,0.694218,0.016160,0.618460,0.529823,0.673566,0.437107,0.5350,14.00,0.047861,0.024844
1,small_dropout50_wd1e3,"(32, 16)",0.50,0.001,0.0008,64,4,0.654712,0.012743,0.638611,...,0.692581,0.018513,0.616774,0.523684,0.667166,0.435535,0.5200,11.25,0.043966,0.020950
0,small_dropout40_wd1e3,"(32, 16)",0.40,0.001,0.0008,64,4,0.652885,0.014936,0.637373,...,0.694978,0.008480,0.612994,0.522364,0.657471,0.436321,0.5225,12.25,0.042139,0.019123
2,stage1_like_stronger_wd,"(64, 32)",0.35,0.001,0.0008,64,4,0.629880,0.036068,0.576410,...,0.676702,0.028097,0.596897,0.528399,0.567998,0.537736,0.4925,2.75,0.019134,-0.003882



=== Test Metrics By Run ===


,experiment_id,config_id,feature_timing,subset_name,model_family,window,split,seed,hidden_dims,dropout,...,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision,model_path
2,presleep_stage1_hp_small_dropout40_wd1e3_seed7,small_dropout40_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,7,"(32, 16)",0.40,...,0.550669,0.702439,0.452830,502,61,174,144,0.702995,0.626676,data\processed\pre_sleep_forecasting\design_c_...
5,presleep_stage1_hp_small_dropout40_wd1e3_seed123,small_dropout40_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,123,"(32, 16)",0.40,...,0.514071,0.637209,0.430818,485,78,181,137,0.688523,0.606593,data\processed\pre_sleep_forecasting\design_c_...
8,presleep_stage1_hp_small_dropout40_wd1e3_seed2026,small_dropout40_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,2026,"(32, 16)",0.40,...,0.541958,0.610236,0.487421,464,99,163,155,0.701565,0.618075,data\processed\pre_sleep_forecasting\design_c_...
11,presleep_stage1_hp_small_dropout40_wd1e3_seed2027,small_dropout40_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,2027,"(32, 16)",0.40,...,0.482759,0.680000,0.374214,507,56,199,119,0.686830,0.600632,data\processed\pre_sleep_forecasting\design_c_...
14,presleep_stage1_hp_small_dropout50_wd1e3_seed7,small_dropout50_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,7,"(32, 16)",0.50,...,0.553734,0.658009,0.477987,484,79,166,152,0.708195,0.631612,data\processed\pre_sleep_forecasting\design_c_...
17,presleep_stage1_hp_small_dropout50_wd1e3_seed123,small_dropout50_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,123,"(32, 16)",0.50,...,0.493151,0.652850,0.396226,496,67,192,126,0.667248,0.598140,data\processed\pre_sleep_forecasting\design_c_...
20,presleep_stage1_hp_small_dropout50_wd1e3_seed2026,small_dropout50_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,2026,"(32, 16)",0.50,...,0.545775,0.620000,0.487421,468,95,163,155,0.704347,0.632911,data\processed\pre_sleep_forecasting\design_c_...
23,presleep_stage1_hp_small_dropout50_wd1e3_seed2027,small_dropout50_wd1e3,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,2027,"(32, 16)",0.50,...,0.502075,0.737805,0.380503,520,43,197,121,0.690534,0.604432,data\processed\pre_sleep_forecasting\design_c_...
38,presleep_stage1_hp_stage1_like_stronger_wd_seed7,stage1_like_stronger_wd,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,7,"(64, 32)",0.35,...,0.524203,0.419660,0.698113,256,307,96,222,0.648888,0.578932,data\processed\pre_sleep_forecasting\design_c_...
41,presleep_stage1_hp_stage1_like_stronger_wd_see...,stage1_like_stronger_wd,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,123,"(64, 32)",0.35,...,0.521101,0.625551,0.446541,478,85,176,142,0.658590,0.567566,data\processed\pre_sleep_forecasting\design_c_...



=== Reference Comparison ===


,reference,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,note,row_type
0,stage1_single_seed,0.633762,0.687501,0.600856,0.490421,0.627451,0.402516,best single Stage 1 run so far,existing_reference
1,stage1_seed_mean,0.610746,0.668134,0.601623,0.530948,0.494221,0.620545,more reliable Stage 1 robustness reference,existing_reference
2,stage2_full_rolling,0.602486,0.662779,0.585455,0.530719,0.454139,0.638365,rolling did not improve generalization,existing_reference
3,stage2b_compact_rolling,0.592293,0.685227,0.578818,0.553005,0.423786,0.795597,compact rolling also did not improve BA,existing_reference
4,best_hp_config__tiny_dropout40_wd1e3,0.658607,0.694218,0.618460,0.529823,0.673566,0.437107,mean across stability seeds,new_best_hyperparameter_mean



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [4]:
# Cell 3. Select representative best-config checkpoint by validation performance
# 원하는 결과:
# - best hyperparameter config인 tiny_dropout40_wd1e3의 seed별 validation/test metrics를 정리한다.
# - 대표 모델은 validation balanced accuracy 기준으로 선택한다.
# - test는 선택된 대표 모델의 held-out 결과로만 보고한다.
# - selection summary를 저장하고 log를 업데이트한다.

BEST_CONFIG_ID = "tiny_dropout40_wd1e3"

SELECTION_SUMMARY_PATH = OUTPUT_DIR / "stage1_best_config_seed_selection_summary.csv"
SELECTION_MARKDOWN_PATH = OUTPUT_DIR / "stage1_best_config_seed_selection_summary.md"

stability_metrics_df = pd.read_csv(METRICS_PATH, encoding="utf-8-sig")
stability_summary_df = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")

best_config_metrics_df = stability_metrics_df[
    stability_metrics_df["config_id"] == BEST_CONFIG_ID
].copy()

best_config_validation_df = best_config_metrics_df[
    best_config_metrics_df["split"] == "validation"
].copy()

best_config_test_df = best_config_metrics_df[
    best_config_metrics_df["split"] == "test"
].copy()

selected_validation_row = (
    best_config_validation_df
    .sort_values(
        ["balanced_accuracy", "roc_auc", "average_precision", "f1"],
        ascending=[False, False, False, False],
    )
    .iloc[0]
)

selected_experiment_id = selected_validation_row["experiment_id"]
selected_seed = int(selected_validation_row["seed"])

selected_test_row = best_config_test_df[
    best_config_test_df["experiment_id"] == selected_experiment_id
].iloc[0]

best_config_summary_row = stability_summary_df[
    stability_summary_df["config_id"] == BEST_CONFIG_ID
].iloc[0]

selection_rows = []

for _, validation_row in best_config_validation_df.iterrows():
    experiment_id = validation_row["experiment_id"]
    test_row = best_config_test_df[
        best_config_test_df["experiment_id"] == experiment_id
    ].iloc[0]

    selection_rows.append(
        {
            "selection_role": (
                "selected_by_validation"
                if experiment_id == selected_experiment_id
                else "not_selected"
            ),
            "experiment_id": experiment_id,
            "config_id": BEST_CONFIG_ID,
            "seed": int(validation_row["seed"]),
            "best_epoch": int(validation_row["best_epoch"]),
            "selected_threshold_from_validation": validation_row["selected_threshold_from_validation"],
            "validation_balanced_accuracy": validation_row["balanced_accuracy"],
            "validation_roc_auc": validation_row["roc_auc"],
            "validation_average_precision": validation_row["average_precision"],
            "validation_f1": validation_row["f1"],
            "validation_precision": validation_row["precision"],
            "validation_recall": validation_row["recall"],
            "test_balanced_accuracy": test_row["balanced_accuracy"],
            "test_roc_auc": test_row["roc_auc"],
            "test_average_precision": test_row["average_precision"],
            "test_f1": test_row["f1"],
            "test_precision": test_row["precision"],
            "test_recall": test_row["recall"],
            "model_path": validation_row["model_path"],
        }
    )

selection_summary_df = pd.DataFrame(selection_rows).sort_values(
    ["selection_role", "validation_balanced_accuracy"],
    ascending=[True, False],
)

selection_summary_df.to_csv(
    SELECTION_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

markdown_lines = [
    "# Stage 1 Best Config Seed Selection Summary",
    "",
    "Representative model selection is based on validation metrics only.",
    "",
    f"- Best config: `{BEST_CONFIG_ID}`",
    f"- Selected experiment: `{selected_experiment_id}`",
    f"- Selected seed: `{selected_seed}`",
    f"- Selection metric: validation balanced accuracy",
    f"- Selected validation balanced accuracy: {selected_validation_row['balanced_accuracy']:.4f}",
    f"- Selected test balanced accuracy: {selected_test_row['balanced_accuracy']:.4f}",
    f"- Selected test ROC AUC: {selected_test_row['roc_auc']:.4f}",
    f"- Selected test average precision: {selected_test_row['average_precision']:.4f}",
    f"- Selected test F1: {selected_test_row['f1']:.4f}",
    f"- Selected test precision: {selected_test_row['precision']:.4f}",
    f"- Selected test recall: {selected_test_row['recall']:.4f}",
    "",
    "Best-config seed robustness:",
    "",
    f"- Runs: {int(best_config_summary_row['runs'])}",
    f"- Mean test balanced accuracy: {best_config_summary_row['mean_test_balanced_accuracy']:.4f}",
    f"- Std test balanced accuracy: {best_config_summary_row['std_test_balanced_accuracy']:.4f}",
    f"- Min test balanced accuracy: {best_config_summary_row['min_test_balanced_accuracy']:.4f}",
    f"- Max test balanced accuracy: {best_config_summary_row['max_test_balanced_accuracy']:.4f}",
    f"- Mean test ROC AUC: {best_config_summary_row['mean_test_roc_auc']:.4f}",
    f"- Mean test average precision: {best_config_summary_row['mean_test_average_precision']:.4f}",
    "",
    "Interpretation:",
    "",
    "- This is the current best strict pre-sleep forecasting candidate.",
    "- It uses only Stage 1 strict pre-sleep features, not rolling/history features.",
    "- The selected representative checkpoint is chosen without looking at test performance.",
]

SELECTION_MARKDOWN_PATH.write_text(
    "\n".join(markdown_lines),
    encoding="utf-8",
)

metadata["stage1_best_config_selection"] = {
    "best_config_id": BEST_CONFIG_ID,
    "selected_experiment_id": selected_experiment_id,
    "selected_seed": selected_seed,
    "selection_metric": "validation_balanced_accuracy",
    "selection_summary_path": str(SELECTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
    "selection_markdown_path": str(SELECTION_MARKDOWN_PATH.relative_to(PROJECT_ROOT)),
    "selected_validation_balanced_accuracy": float(selected_validation_row["balanced_accuracy"]),
    "selected_test_balanced_accuracy": float(selected_test_row["balanced_accuracy"]),
    "best_config_mean_test_balanced_accuracy": float(best_config_summary_row["mean_test_balanced_accuracy"]),
    "best_config_std_test_balanced_accuracy": float(best_config_summary_row["std_test_balanced_accuracy"]),
}

STAGE1_METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep Stage 1 best config seed selection

Selected representative checkpoint for best Stage 1 hyperparameter config using validation only.

- Best config: `{BEST_CONFIG_ID}`
- Selected experiment: `{selected_experiment_id}`
- Selected seed: {selected_seed}
- Selection metric: validation balanced accuracy
- Selected validation balanced accuracy: {selected_validation_row["balanced_accuracy"]:.4f}
- Selected test balanced accuracy: {selected_test_row["balanced_accuracy"]:.4f}
- Selected test ROC AUC: {selected_test_row["roc_auc"]:.4f}
- Selected test average precision: {selected_test_row["average_precision"]:.4f}
- Selected test F1: {selected_test_row["f1"]:.4f}
- Selected test precision: {selected_test_row["precision"]:.4f}
- Selected test recall: {selected_test_row["recall"]:.4f}
- Best-config mean test balanced accuracy across seeds: {best_config_summary_row["mean_test_balanced_accuracy"]:.4f}
- Best-config std test balanced accuracy across seeds: {best_config_summary_row["std_test_balanced_accuracy"]:.4f}
- Selection summary: `{SELECTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- Markdown summary: `{SELECTION_MARKDOWN_PATH.relative_to(PROJECT_ROOT)}`
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Best Config Seed Selection Saved ===")
print("selection summary:", SELECTION_SUMMARY_PATH.relative_to(PROJECT_ROOT), SELECTION_SUMMARY_PATH.exists())
print("markdown summary:", SELECTION_MARKDOWN_PATH.relative_to(PROJECT_ROOT), SELECTION_MARKDOWN_PATH.exists())

print("\n=== Best Config Seed Selection Summary ===")
display(selection_summary_df)

print("\n=== Selected Representative Model ===")
display(
    pd.DataFrame(
        [
            {
                "selected_experiment_id": selected_experiment_id,
                "selected_seed": selected_seed,
                "config_id": BEST_CONFIG_ID,
                "best_epoch": int(selected_validation_row["best_epoch"]),
                "threshold": selected_validation_row["selected_threshold_from_validation"],
                "validation_balanced_accuracy": selected_validation_row["balanced_accuracy"],
                "test_balanced_accuracy": selected_test_row["balanced_accuracy"],
                "test_roc_auc": selected_test_row["roc_auc"],
                "test_average_precision": selected_test_row["average_precision"],
                "test_f1": selected_test_row["f1"],
                "test_precision": selected_test_row["precision"],
                "test_recall": selected_test_row["recall"],
                "model_path": selected_validation_row["model_path"],
            }
        ]
    )
)

print("\n=== Best Config Robustness Summary ===")
display(
    pd.DataFrame(
        [
            {
                "config_id": BEST_CONFIG_ID,
                "runs": int(best_config_summary_row["runs"]),
                "mean_test_balanced_accuracy": best_config_summary_row["mean_test_balanced_accuracy"],
                "std_test_balanced_accuracy": best_config_summary_row["std_test_balanced_accuracy"],
                "min_test_balanced_accuracy": best_config_summary_row["min_test_balanced_accuracy"],
                "max_test_balanced_accuracy": best_config_summary_row["max_test_balanced_accuracy"],
                "mean_test_roc_auc": best_config_summary_row["mean_test_roc_auc"],
                "mean_test_average_precision": best_config_summary_row["mean_test_average_precision"],
            }
        ]
    )
)

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Best Config Seed Selection Saved ===
selection summary: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_hyperparameter_stability_outputs\stage1_best_config_seed_selection_summary.csv True
markdown summary: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_hyperparameter_stability_outputs\stage1_best_config_seed_selection_summary.md True

=== Best Config Seed Selection Summary ===


,selection_role,experiment_id,config_id,seed,best_epoch,selected_threshold_from_validation,validation_balanced_accuracy,validation_roc_auc,validation_average_precision,validation_f1,validation_precision,validation_recall,test_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,model_path
0,not_selected,presleep_stage1_hp_tiny_dropout40_wd1e3_seed7,tiny_dropout40_wd1e3,7,22,0.52,0.674390,0.691876,0.518657,0.582677,0.560606,0.606557,0.667669,0.710008,0.625793,0.549815,0.665179,0.468553,data\processed\pre_sleep_forecasting\design_c_...
2,not_selected,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2026,tiny_dropout40_wd1e3,2026,17,0.57,0.668761,0.680583,0.515596,0.572581,0.563492,0.581967,0.661031,0.701029,0.625187,0.531792,0.686567,0.433962,data\processed\pre_sleep_forecasting\design_c_...
1,not_selected,presleep_stage1_hp_tiny_dropout40_wd1e3_seed123,tiny_dropout40_wd1e3,123,5,0.51,0.668069,0.661821,0.513345,0.574803,0.553030,0.598361,0.656518,0.672140,0.604174,0.522417,0.687179,0.421384,data\processed\pre_sleep_forecasting\design_c_...
3,selected_by_validation,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,tiny_dropout40_wd1e3,2027,12,0.54,0.676958,0.682368,0.512725,0.584000,0.570312,0.598361,0.649209,0.693695,0.618684,0.515267,0.655340,0.424528,data\processed\pre_sleep_forecasting\design_c_...



=== Selected Representative Model ===


,selected_experiment_id,selected_seed,config_id,best_epoch,threshold,validation_balanced_accuracy,test_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,model_path
0,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,2027,tiny_dropout40_wd1e3,12,0.54,0.676958,0.649209,0.693695,0.618684,0.515267,0.65534,0.424528,data\processed\pre_sleep_forecasting\design_c_...



=== Best Config Robustness Summary ===


,config_id,runs,mean_test_balanced_accuracy,std_test_balanced_accuracy,min_test_balanced_accuracy,max_test_balanced_accuracy,mean_test_roc_auc,mean_test_average_precision
0,tiny_dropout40_wd1e3,4,0.658607,0.007761,0.649209,0.667669,0.694218,0.61846



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [5]:
# Cell 4. Threshold sensitivity for selected Stage 1 best-config representative model
# 원하는 결과:
# - selected representative model의 prediction을 사용해 threshold별 validation/test 성능을 계산한다.
# - validation 기준 threshold 정책들을 비교한다.
# - test-only best threshold는 diagnostic only로 분리한다.
# - threshold sensitivity 결과를 저장하고 log를 업데이트한다.

SELECTED_EXPERIMENT_ID = selected_experiment_id

SELECTED_THRESHOLD_SENSITIVITY_PATH = OUTPUT_DIR / "stage1_selected_model_threshold_sensitivity.csv"
SELECTED_THRESHOLD_POLICY_PATH = OUTPUT_DIR / "stage1_selected_model_threshold_policy_comparison.csv"
SELECTED_TEST_DIAGNOSTIC_PATH = OUTPUT_DIR / "stage1_selected_model_test_threshold_diagnostic.csv"

stability_predictions_df = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")

selected_predictions_df = stability_predictions_df[
    stability_predictions_df["experiment_id"] == SELECTED_EXPERIMENT_ID
].copy()

threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

sensitivity_rows = []

for split_name in ["train", "validation", "test"]:
    split_predictions = selected_predictions_df[
        selected_predictions_df["split"] == split_name
    ].copy()

    y_true = split_predictions["y_true"].to_numpy()
    y_probability = split_predictions["y_probability"].to_numpy()

    for threshold in threshold_grid:
        metrics = evaluate_binary_predictions(
            y_true=y_true,
            y_probability=y_probability,
            threshold=threshold,
        )
        sensitivity_rows.append(
            {
                "experiment_id": SELECTED_EXPERIMENT_ID,
                "split": split_name,
                **metrics,
            }
        )

selected_threshold_sensitivity_df = pd.DataFrame(sensitivity_rows)

validation_sensitivity_df = selected_threshold_sensitivity_df[
    selected_threshold_sensitivity_df["split"] == "validation"
].copy()

test_sensitivity_df = selected_threshold_sensitivity_df[
    selected_threshold_sensitivity_df["split"] == "test"
].copy()

def select_threshold_policy(validation_df, policy_name):
    if policy_name == "validation_balanced_accuracy":
        return (
            validation_df
            .sort_values(["balanced_accuracy", "f1", "precision"], ascending=[False, False, False])
            .iloc[0]
        )

    if policy_name == "validation_f1":
        return (
            validation_df
            .sort_values(["f1", "balanced_accuracy", "precision"], ascending=[False, False, False])
            .iloc[0]
        )

    if policy_name == "validation_recall_priority_precision_at_least_0_50":
        eligible = validation_df[validation_df["precision"] >= 0.50].copy()
        if len(eligible) == 0:
            eligible = validation_df.copy()
        return (
            eligible
            .sort_values(["recall", "balanced_accuracy", "f1"], ascending=[False, False, False])
            .iloc[0]
        )

    if policy_name == "validation_precision_priority_recall_at_least_0_40":
        eligible = validation_df[validation_df["recall"] >= 0.40].copy()
        if len(eligible) == 0:
            eligible = validation_df.copy()
        return (
            eligible
            .sort_values(["precision", "balanced_accuracy", "f1"], ascending=[False, False, False])
            .iloc[0]
        )

    raise ValueError(f"Unsupported policy: {policy_name}")

threshold_policies = [
    "validation_balanced_accuracy",
    "validation_f1",
    "validation_recall_priority_precision_at_least_0_50",
    "validation_precision_priority_recall_at_least_0_40",
]

policy_rows = []

for policy_name in threshold_policies:
    selected_validation_row = select_threshold_policy(
        validation_sensitivity_df,
        policy_name,
    )
    selected_threshold = float(selected_validation_row["threshold"])

    for split_name in ["validation", "test"]:
        split_row = selected_threshold_sensitivity_df[
            (selected_threshold_sensitivity_df["split"] == split_name)
            & (selected_threshold_sensitivity_df["threshold"] == selected_threshold)
        ].iloc[0]

        policy_rows.append(
            {
                "experiment_id": SELECTED_EXPERIMENT_ID,
                "threshold_policy": policy_name,
                "selected_threshold": selected_threshold,
                "evaluation_split": split_name,
                "balanced_accuracy": split_row["balanced_accuracy"],
                "roc_auc": split_row["roc_auc"],
                "average_precision": split_row["average_precision"],
                "f1": split_row["f1"],
                "precision": split_row["precision"],
                "recall": split_row["recall"],
                "tn": int(split_row["tn"]),
                "fp": int(split_row["fp"]),
                "fn": int(split_row["fn"]),
                "tp": int(split_row["tp"]),
            }
        )

selected_threshold_policy_df = pd.DataFrame(policy_rows)

test_diagnostic_rows = []

for metric_name in ["balanced_accuracy", "f1", "precision", "recall"]:
    if metric_name == "recall":
        diagnostic_pool = test_sensitivity_df[test_sensitivity_df["precision"] >= 0.30].copy()
        diagnostic_label = "test_recall_with_precision_at_least_0_30"
        if len(diagnostic_pool) == 0:
            diagnostic_pool = test_sensitivity_df.copy()
        diagnostic_row = (
            diagnostic_pool
            .sort_values(["recall", "balanced_accuracy"], ascending=[False, False])
            .iloc[0]
        )
    else:
        diagnostic_label = f"test_best_{metric_name}"
        diagnostic_row = (
            test_sensitivity_df
            .sort_values([metric_name, "balanced_accuracy"], ascending=[False, False])
            .iloc[0]
        )

    test_diagnostic_rows.append(
        {
            "experiment_id": SELECTED_EXPERIMENT_ID,
            "diagnostic_policy": diagnostic_label,
            "threshold": diagnostic_row["threshold"],
            "balanced_accuracy": diagnostic_row["balanced_accuracy"],
            "roc_auc": diagnostic_row["roc_auc"],
            "average_precision": diagnostic_row["average_precision"],
            "f1": diagnostic_row["f1"],
            "precision": diagnostic_row["precision"],
            "recall": diagnostic_row["recall"],
            "tn": int(diagnostic_row["tn"]),
            "fp": int(diagnostic_row["fp"]),
            "fn": int(diagnostic_row["fn"]),
            "tp": int(diagnostic_row["tp"]),
        }
    )

test_diagnostic_df = pd.DataFrame(test_diagnostic_rows)

selected_threshold_sensitivity_df.to_csv(
    SELECTED_THRESHOLD_SENSITIVITY_PATH,
    index=False,
    encoding="utf-8-sig",
)
selected_threshold_policy_df.to_csv(
    SELECTED_THRESHOLD_POLICY_PATH,
    index=False,
    encoding="utf-8-sig",
)
test_diagnostic_df.to_csv(
    SELECTED_TEST_DIAGNOSTIC_PATH,
    index=False,
    encoding="utf-8-sig",
)

metadata["stage1_selected_model_threshold_sensitivity"] = {
    "selected_experiment_id": SELECTED_EXPERIMENT_ID,
    "threshold_sensitivity_path": str(SELECTED_THRESHOLD_SENSITIVITY_PATH.relative_to(PROJECT_ROOT)),
    "threshold_policy_path": str(SELECTED_THRESHOLD_POLICY_PATH.relative_to(PROJECT_ROOT)),
    "test_diagnostic_path": str(SELECTED_TEST_DIAGNOSTIC_PATH.relative_to(PROJECT_ROOT)),
    "threshold_policies": threshold_policies,
    "note": "Threshold policies are selected on validation only. Test-best rows are diagnostic only.",
}

STAGE1_METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

validation_balanced_policy_test_row = selected_threshold_policy_df[
    (selected_threshold_policy_df["threshold_policy"] == "validation_balanced_accuracy")
    & (selected_threshold_policy_df["evaluation_split"] == "test")
].iloc[0]

log_entry = f"""

## 2026-06-29 - Selected pre-sleep Stage 1 model threshold sensitivity

Evaluated threshold sensitivity for selected representative Stage 1 model.

- Selected experiment: `{SELECTED_EXPERIMENT_ID}`
- Threshold sensitivity: `{SELECTED_THRESHOLD_SENSITIVITY_PATH.relative_to(PROJECT_ROOT)}`
- Threshold policy comparison: `{SELECTED_THRESHOLD_POLICY_PATH.relative_to(PROJECT_ROOT)}`
- Test diagnostic thresholds: `{SELECTED_TEST_DIAGNOSTIC_PATH.relative_to(PROJECT_ROOT)}`
- Validation-balanced policy threshold: {validation_balanced_policy_test_row["selected_threshold"]:.2f}
- Validation-balanced policy test balanced accuracy: {validation_balanced_policy_test_row["balanced_accuracy"]:.4f}
- Validation-balanced policy test precision: {validation_balanced_policy_test_row["precision"]:.4f}
- Validation-balanced policy test recall: {validation_balanced_policy_test_row["recall"]:.4f}
- Test-best threshold rows are diagnostic only and are not used for model selection.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Selected Model Threshold Sensitivity Saved ===")
print("threshold sensitivity:", SELECTED_THRESHOLD_SENSITIVITY_PATH.relative_to(PROJECT_ROOT), SELECTED_THRESHOLD_SENSITIVITY_PATH.exists())
print("threshold policy:", SELECTED_THRESHOLD_POLICY_PATH.relative_to(PROJECT_ROOT), SELECTED_THRESHOLD_POLICY_PATH.exists())
print("test diagnostic:", SELECTED_TEST_DIAGNOSTIC_PATH.relative_to(PROJECT_ROOT), SELECTED_TEST_DIAGNOSTIC_PATH.exists())

print("\n=== Threshold Policy Comparison: Validation Rows ===")
display(
    selected_threshold_policy_df[
        selected_threshold_policy_df["evaluation_split"] == "validation"
    ]
)

print("\n=== Threshold Policy Comparison: Test Rows ===")
display(
    selected_threshold_policy_df[
        selected_threshold_policy_df["evaluation_split"] == "test"
    ]
)

print("\n=== Test Threshold Diagnostic Only ===")
display(test_diagnostic_df)

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Selected Model Threshold Sensitivity Saved ===
threshold sensitivity: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_hyperparameter_stability_outputs\stage1_selected_model_threshold_sensitivity.csv True
threshold policy: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_hyperparameter_stability_outputs\stage1_selected_model_threshold_policy_comparison.csv True
test diagnostic: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_hyperparameter_stability_outputs\stage1_selected_model_test_threshold_diagnostic.csv True

=== Threshold Policy Comparison: Validation Rows ===


,experiment_id,threshold_policy,selected_threshold,evaluation_split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
0,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_balanced_accuracy,0.54,validation,0.676958,0.682368,0.512725,0.584000,0.570312,0.598361,170,55,49,73
2,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_f1,0.54,validation,0.676958,0.682368,0.512725,0.584000,0.570312,0.598361,170,55,49,73
4,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_recall_priority_precision_at_least_...,0.47,validation,0.650437,0.682368,0.512725,0.566308,0.503185,0.647541,147,78,43,79
6,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_precision_priority_recall_at_least_...,0.58,validation,0.654098,0.682368,0.512725,0.541485,0.579439,0.508197,180,45,60,62



=== Threshold Policy Comparison: Test Rows ===


,experiment_id,threshold_policy,selected_threshold,evaluation_split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
1,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_balanced_accuracy,0.54,test,0.649209,0.693695,0.618684,0.515267,0.65534,0.424528,492,71,183,135
3,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_f1,0.54,test,0.649209,0.693695,0.618684,0.515267,0.65534,0.424528,492,71,183,135
5,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_recall_priority_precision_at_least_...,0.47,test,0.659165,0.693695,0.618684,0.566096,0.56000,0.572327,420,143,136,182
7,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,validation_precision_priority_recall_at_least_...,0.58,test,0.622728,0.693695,0.618684,0.428894,0.76000,0.298742,533,30,223,95



=== Test Threshold Diagnostic Only ===


,experiment_id,diagnostic_policy,threshold,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
0,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,test_best_balanced_accuracy,0.47,0.659165,0.693695,0.618684,0.566096,0.560000,0.572327,420,143,136,182
1,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,test_best_f1,0.36,0.606036,0.693695,0.618684,0.567742,0.431373,0.830189,215,348,54,264
2,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,test_best_precision,0.74,0.507862,0.693695,0.618684,0.030960,1.000000,0.015723,563,0,313,5
3,presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027,test_recall_with_precision_at_least_0_30,0.05,0.500000,0.693695,0.618684,0.530442,0.360953,1.000000,0,563,0,318



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [6]:
# Cell 5. Write final pre-sleep forecasting report
# 원하는 결과:
# - pre-sleep forecasting 전체 결과를 최종 리포트로 정리한다.
# - Design C dataset, Stage 1/2/2B, hyperparameter stability, threshold policy를 요약한다.
# - 현재 최종 대표 모델과 한계를 명확히 쓴다.
# - reports/pre_sleep_forecasting_stage1_final_report.md 저장 및 log 업데이트.

REPORT_PATH = PROJECT_ROOT / "reports" / "pre_sleep_forecasting_stage1_final_report.md"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

reference_df = pd.read_csv(COMPARISON_PATH, encoding="utf-8-sig")
selection_summary_df = pd.read_csv(SELECTION_SUMMARY_PATH, encoding="utf-8-sig")
threshold_policy_df = pd.read_csv(SELECTED_THRESHOLD_POLICY_PATH, encoding="utf-8-sig")
threshold_diagnostic_df = pd.read_csv(SELECTED_TEST_DIAGNOSTIC_PATH, encoding="utf-8-sig")
stability_summary_df = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")

best_config_row = stability_summary_df[
    stability_summary_df["config_id"] == BEST_CONFIG_ID
].iloc[0]

selected_row = selection_summary_df[
    selection_summary_df["selection_role"] == "selected_by_validation"
].iloc[0]

official_test_policy_row = threshold_policy_df[
    (threshold_policy_df["threshold_policy"] == "validation_balanced_accuracy")
    & (threshold_policy_df["evaluation_split"] == "test")
].iloc[0]

recall_policy_test_row = threshold_policy_df[
    (threshold_policy_df["threshold_policy"] == "validation_recall_priority_precision_at_least_0_50")
    & (threshold_policy_df["evaluation_split"] == "test")
].iloc[0]

stage1_seed_mean_row = reference_df[
    reference_df["reference"] == "stage1_seed_mean"
].iloc[0]

stage2_row = reference_df[
    reference_df["reference"] == "stage2_full_rolling"
].iloc[0]

stage2b_row = reference_df[
    reference_df["reference"] == "stage2b_compact_rolling"
].iloc[0]

report = f"""# Pre-Sleep Forecasting Final Report

## 1. Objective

The prediction objective was redefined as strict pre-sleep forecasting:

> Predict the quality label of the upcoming sleep episode using only wearable-derived data available before `sleep_start_datetime`.

This differs from earlier same-date classification. Earlier same-date models can perform well, but they are not aligned with the intended real-use timing because `calendar_date` in the sleep target data corresponds to sleep end date.

## 2. Dataset Design

The final forecasting unit is one sleep episode.

- Rows: 3,551 sleep episodes
- Participants: 69
- Target: `good_sleep_label`
- Positive class ratio: approximately 0.3937
- Prediction cutoff: `sleep_start_datetime`
- Split strategy: participant-level train/validation/test split
- Train / validation / test samples: 2,323 / 347 / 881
- Train / validation / test participants: 46 / 9 / 14

Strict Design C features were created from:

- MongoDB intraday steps before sleep onset
- MongoDB intraday calories before sleep onset
- MongoDB intraday heart rate before sleep onset
- Sleep start timing/calendar features known before prediction
- Previous-day daily activity/resting-HR features

MongoDB server-side aggregation was validated against raw-fetch aggregation on sample episodes with zero mismatched features.

## 3. Feature Sets Tested

### Stage 1: Strict Pre-Sleep Features

Stage 1 used the core strict pre-sleep feature set.

- Final features: 58
- Feature groups:
  - pre-sleep intraday
  - previous-day daily
  - timing/calendar
  - missing indicators
- No rolling/history features

### Stage 2: Full Rolling/History Features

Stage 2 added broad prior-episode rolling/history features.

- Final features: 380
- Rolling/history features: 319
- Result: did not improve held-out test generalization

### Stage 2B: Compact Rolling/History Features

Stage 2B reduced rolling features to compact prior-episode summaries.

- Final features: 148
- Compact rolling features: 87
- Result: improved recall but did not improve balanced accuracy

## 4. Model Experiments

All models in the pre-sleep workflow were neural-network MLP models trained in PyTorch.

Traditional ML baselines were not used for final model selection in this stage.

### Main Results

| Candidate | Feature set | Test balanced accuracy | ROC AUC | Average precision | F1 | Precision | Recall |
|---|---:|---:|---:|---:|---:|---:|---:|
| Stage 1 single seed | 58 | 0.6338 | 0.6875 | 0.6009 | 0.4904 | 0.6275 | 0.4025 |
| Stage 1 seed mean | 58 | {stage1_seed_mean_row['balanced_accuracy']:.4f} | {stage1_seed_mean_row['roc_auc']:.4f} | {stage1_seed_mean_row['average_precision']:.4f} | {stage1_seed_mean_row['f1']:.4f} | {stage1_seed_mean_row['precision']:.4f} | {stage1_seed_mean_row['recall']:.4f} |
| Stage 2 full rolling | 380 | {stage2_row['balanced_accuracy']:.4f} | {stage2_row['roc_auc']:.4f} | {stage2_row['average_precision']:.4f} | {stage2_row['f1']:.4f} | {stage2_row['precision']:.4f} | {stage2_row['recall']:.4f} |
| Stage 2B compact rolling | 148 | {stage2b_row['balanced_accuracy']:.4f} | {stage2b_row['roc_auc']:.4f} | {stage2b_row['average_precision']:.4f} | {stage2b_row['f1']:.4f} | {stage2b_row['precision']:.4f} | {stage2b_row['recall']:.4f} |
| Best Stage 1 HP config mean | 58 | {best_config_row['mean_test_balanced_accuracy']:.4f} | {best_config_row['mean_test_roc_auc']:.4f} | {best_config_row['mean_test_average_precision']:.4f} | {best_config_row['mean_test_f1']:.4f} | {best_config_row['mean_test_precision']:.4f} | {best_config_row['mean_test_recall']:.4f} |

## 5. Best Hyperparameter Configuration

The best stability configuration was:

- Config: `{BEST_CONFIG_ID}`
- Hidden dimensions: `(24, 12)`
- Dropout: 0.40
- Weight decay: 0.001
- Learning rate: 0.0008
- Seeds: 7, 123, 2026, 2027

Seed robustness for this config:

- Mean test balanced accuracy: {best_config_row['mean_test_balanced_accuracy']:.4f}
- Std test balanced accuracy: {best_config_row['std_test_balanced_accuracy']:.4f}
- Min / max test balanced accuracy: {best_config_row['min_test_balanced_accuracy']:.4f} / {best_config_row['max_test_balanced_accuracy']:.4f}
- Mean ROC AUC: {best_config_row['mean_test_roc_auc']:.4f}
- Mean average precision: {best_config_row['mean_test_average_precision']:.4f}

This configuration improved over the previous Stage 1 seed mean by {best_config_row['delta_mean_ba_vs_stage1_seed_mean']:.4f} balanced accuracy.

## 6. Selected Representative Model

The representative checkpoint was selected by validation balanced accuracy only.

- Selected experiment: `{selected_row['experiment_id']}`
- Selected seed: {int(selected_row['seed'])}
- Selected threshold: {selected_row['selected_threshold_from_validation']:.2f}
- Validation balanced accuracy: {selected_row['validation_balanced_accuracy']:.4f}

Held-out test performance for the selected representative model:

- Test balanced accuracy: {selected_row['test_balanced_accuracy']:.4f}
- Test ROC AUC: {selected_row['test_roc_auc']:.4f}
- Test average precision: {selected_row['test_average_precision']:.4f}
- Test F1: {selected_row['test_f1']:.4f}
- Test precision: {selected_row['test_precision']:.4f}
- Test recall: {selected_row['test_recall']:.4f}

## 7. Threshold Policy

The official threshold policy is validation balanced accuracy.

Official policy:

- Threshold: {official_test_policy_row['selected_threshold']:.2f}
- Test balanced accuracy: {official_test_policy_row['balanced_accuracy']:.4f}
- Test precision: {official_test_policy_row['precision']:.4f}
- Test recall: {official_test_policy_row['recall']:.4f}

A recall-priority validation policy is also available as an operating option:

- Threshold: {recall_policy_test_row['selected_threshold']:.2f}
- Test balanced accuracy: {recall_policy_test_row['balanced_accuracy']:.4f}
- Test precision: {recall_policy_test_row['precision']:.4f}
- Test recall: {recall_policy_test_row['recall']:.4f}

The recall-priority policy may be useful if the application values catching more likely good-sleep episodes, but it produces more false positives. It should be presented as an alternative operating policy, not as the primary model-selection result.

## 8. Interpretation

The best current model is a strict pre-sleep forecasting MLP using Stage 1 Design C features.

The model shows meaningful predictive signal:

- It outperforms the earlier previous-day reference in balanced accuracy.
- It is aligned with the intended real-use objective.
- The best hyperparameter configuration is stable across seeds.
- ROC AUC and average precision are moderate, not high.

The model should be considered useful for exploratory decision support or personal feedback research, but not yet reliable enough for high-stakes health decision-making.

## 9. Limitations

Important limitations remain:

- Test performance is moderate.
- Recall under the official threshold is limited.
- The validation split is relatively small.
- Participants are heterogeneous, and generalization to unseen people remains challenging.
- Rolling/history features did not improve generalization, likely due to overfitting or participant-specific noise.
- No external validation dataset has been used.

## 10. Current Recommendation

Use the selected Stage 1 hyperparameter-stabilized MLP as the current best pre-sleep forecasting candidate.

Current best candidate:

- Experiment family: `presleep_stage1_hp_tiny_dropout40_wd1e3`
- Representative model: `{selected_row['experiment_id']}`
- Feature set: strict pre-sleep Stage 1
- Model: tiny regularized MLP
- Official threshold: {official_test_policy_row['selected_threshold']:.2f}
- Representative test balanced accuracy: {selected_row['test_balanced_accuracy']:.4f}
- Robustness mean test balanced accuracy: {best_config_row['mean_test_balanced_accuracy']:.4f}

Recommended next work:

1. Participant-level bootstrap confidence interval for the selected model.
2. Calibration analysis.
3. Optional sequence model using strict pre-sleep episode sequences.
4. External or future-period validation if more data becomes available.
"""

REPORT_PATH.write_text(report, encoding="utf-8")

log_entry = f"""

## 2026-06-29 - Pre-sleep forecasting final report

Created final pre-sleep forecasting report.

- Report: `{REPORT_PATH.relative_to(PROJECT_ROOT)}`
- Current best candidate family: `{BEST_CONFIG_ID}`
- Selected representative model: `{selected_row['experiment_id']}`
- Representative test balanced accuracy: {selected_row['test_balanced_accuracy']:.4f}
- Best-config mean test balanced accuracy: {best_config_row['mean_test_balanced_accuracy']:.4f}
- Best-config std test balanced accuracy: {best_config_row['std_test_balanced_accuracy']:.4f}
- Official threshold policy: validation balanced accuracy
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("report written:", REPORT_PATH.relative_to(PROJECT_ROOT), REPORT_PATH.exists())
print("report characters:", len(report))
print("log updated:", LOG_PATH, LOG_PATH.exists())

report written: reports\pre_sleep_forecasting_stage1_final_report.md True
report characters: 6321
log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
